In [31]:
import openai, json
# url을 통해 API를 얻어오기 위해 추가
from urllib.request import urlopen, Request

client = openai.OpenAI()
messages = []

In [47]:
# 지역 날씨 가져오기
def get_weather(city):
    return "33 degree"

# 인기 영화 리스트 가져오기
def get_popular_movies():
    url = "https://nomad-movies-2.nomadcoders.workers.dev/movies"

    # 이 부분은 GPT에게 문의
    # 추후 웹스크랩퍼 강의를 들어야할 것 같다
    request = Request(
        url,
        headers={
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            ),
            "Accept": "application/json",
        },
    )
    with urlopen(request) as response:
        body = response.read().decode("utf-8")
        return json.loads(body)
    
# 영화 세부 정보 가져오기
# 위 함수에서 id만 받아서 넣어주면 되는 구조
def get_movie_details(id):
    # {}에 id를 저렇게 넣어도 된다는 게 굉장히 신기하다
    url = f"https://nomad-movies-2.nomadcoders.workers.dev/movies/{id}"

    # 이 부분은 GPT에게 문의
    # 추후 웹스크랩퍼 강의를 들어야할 것 같다
    request = Request(
        url,
        headers={
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            ),
            "Accept": "application/json",
        },
    )
    with urlopen(request) as response:
        body = response.read().decode("utf-8")
        return json.loads(body)


def get_movie_credits(id):
    # {}에 id를 저렇게 넣어도 된다는 게 굉장히 신기하다
    url = f"https://nomad-movies-2.nomadcoders.workers.dev/movies/{id}/credits"

    # 이 부분은 GPT에게 문의
    # 추후 웹스크랩퍼 강의를 들어야할 것 같다
    request = Request(
        url,
        headers={
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            ),
            "Accept": "application/json",
        },
    )
    with urlopen(request) as response:
        body = response.read().decode("utf-8")
        return json.loads(body)

FUNCTION_MAP = {
    "get_weather": get_weather,
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}

In [46]:
TOOLS = [
    {
        "type": "function",
        "function":{
            "name": "get_weather",
            "description": "A function to get the weather of a city.",
            "parameters":{
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city to get the weather of.",
                    }
                },
                "required": ["city"],
            }
        }
    },
    {
        "type": "function",
        "function":{
            "name": "get_popular_movies",
            "description": "A function to get the popular movies.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            }
        }
    },
    {
        "type": "function",
        "function":{
            "name": "get_movie_credits",
            "description": "A function to get the movie credits by id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id":{
                        "type": "integer",
                        "description": "The id of the movie to get the credits of movie.",
                    }
                },
                "required": ["id"],
            }
        }
    },
    {
        "type": "function",
        "function":{
            "name": "get_movie_details",
            "description": "A function to get the movie detail by id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id":{
                        "type": "integer",
                        "description": "The id of the movie to get the detail of movie.",
                    }
                },
                "required": ["id"],
            }
        }
    },
]

In [34]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):
    # 메모리에 저장하기
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls":[
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function":{
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        }
                    } 
                    for tool_call in message.tool_calls
                ],
            }
        )
        # 실제 실행
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")
            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)
            result = function_to_run(**arguments)
            print(f"Ran {function_name} with args {arguments} for a result of {result}")

            # 얻어온 데이터를 다시 스트링으로 변경해야 요청할 수 있다
            if isinstance(result, str):
                tool_content = result
            else:
                tool_content = json.dumps(result, ensure_ascii=False)
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": tool_content,
            })
        # 응답 메모리에 저장하기
        call_ai();
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")

def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    # AI의 응답을 통해 도구를 사용한다면 이 곳에서 처리
    process_ai_response(response.choices[0].message)
    
    

In [48]:
while True:
    message = input("Send a message to the LLM")
    if message =="quit" or message == "q":
        break
    else:
        messages.append({"role":"user", "content": message})
        print(f"User: {message}")
        call_ai()

User: movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘
Calling function: get_movie_credits with {"id":550}
Ran get_movie_credits with args {'id': 550} for a result of [{'adult': False, 'gender': 2, 'id': 819, 'known_for_department': 'Acting', 'name': 'Edward Norton', 'original_name': 'Edward Norton', 'popularity': 4.6098, 'profile_path': 'https://image.tmdb.org/t/p/w185/8nytsqL59SFJTVYVrN72k6qkGgJ.jpg', 'cast_id': 4, 'character': 'Narrator', 'credit_id': '52fe4250c3a36847f80149f3', 'order': 0}, {'adult': False, 'gender': 2, 'id': 287, 'known_for_department': 'Acting', 'name': 'Brad Pitt', 'original_name': 'Brad Pitt', 'popularity': 9.8187, 'profile_path': 'https://image.tmdb.org/t/p/w185/m09Y1YfPPeNYYUSHnnVqahkrC1o.jpg', 'cast_id': 5, 'character': 'Tyler Durden', 'credit_id': '52fe4250c3a36847f80149f7', 'order': 1}, {'adult': False, 'gender': 1, 'id': 1283, 'known_for_department': 'Acting', 'name': 'Helena Bonham Carter', 'original_name': 'Helena Bonham Carter', 'popularity': 4.6027, 'profile_path'